In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)


In [ ]:
print(len(train_loader))


In [ ]:
import matplotlib.pyplot as plt

images, labels = next(iter(train_loader))

plt.imshow(images[55].reshape(28, 28), cmap='gray')
plt.title(f"Label: {labels[0]}")
plt.show()

In [12]:
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 128)

        self.fc2 = nn.Linear(128, 64)

        self.fc3 = nn.Linear(64, 10)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 28 * 28)

        x = self.relu(self.fc1(x))

        x = self.relu(self.fc2(x))

        x = self.fc3(x)
        return x

model = SimpleNet()
print(model)

SimpleNet(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=10, bias=True)
  (relu): ReLU()
)


In [13]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [14]:
epochs = 3 # تعداد دفعاتی که مدل کل دیتاست را مرور می‌کند

for epoch in range(epochs):
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(images)

        # ۳. محاسبه خطا توسط داور
        loss = criterion(outputs, labels)

        # ۴. برگشت به عقب: محاسبه اینکه هر وزن چقدر در خطا مقصر بوده (Backpropagation)
        loss.backward()

        # ۵. آپدیت کردن وزن‌ها توسط مربی
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1} - Loss: {running_loss/len(train_loader):.4f}")

print("آموزش تمام شد! 🎉")

Epoch 1 - Loss: 0.3326
Epoch 2 - Loss: 0.1392
Epoch 3 - Loss: 0.0957
آموزش تمام شد! 🎉


In [15]:
correct = 0
total = 0
with torch.no_grad(): # در زمان تست نیازی به محاسبه مشتق نداریم
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"دقت مدل روی داده‌های جدید: {100 * correct / total:.2f}%")

دقت مدل روی داده‌های جدید: 96.89%
